# 03 — Modelling

Binary classification of cardiac rhythm: Normal (0) vs Abnormal (1).

**Calls:** `src/evaluate.py`
**Input:** `data/processed/physionet_features.csv` (8,187 records × 8 features)
**Output:** `outputs/models/` (saved model + selection report)

**Pre-registered success thresholds** (locked before training):
- Sensitivity ≥ 0.80
- Specificity ≥ 0.75

**Models:** Logistic Regression, Random Forest, XGBoost, SVM
**Split:** 80/10/10 stratified (train / validation / test)

## Load Feature Matrix

Reads the feature matrix produced by `02_feature_engineering.ipynb`. Separates features, binary labels, and original Physionet labels (needed for AF-specific secondary analysis in evaluation).

**Input:** `data/processed/physionet_features.csv`

In [3]:
import sys
import os
import pandas as pd
import numpy as np

sys.path.append(r'C:\Projects\GA Capstone Project')

PROC_DIR = r'C:\Projects\GA Capstone Project\data\processed'

df = pd.read_csv(os.path.join(PROC_DIR, 'physionet_features.csv'))

print(f"Feature matrix loaded: {df.shape[0]:,} records × {df.shape[1]} columns")
print(f"\nClass distribution:")
print(f"  Normal   (0): {(df['binary_label']==0).sum():,} ({(df['binary_label']==0).mean()*100:.1f}%)")
print(f"  Abnormal (1): {(df['binary_label']==1).sum():,} ({(df['binary_label']==1).mean()*100:.1f}%)")
print(f"\nOriginal label breakdown:")
print(df['label'].value_counts().to_string())

# ── Separate features, labels, and metadata ────────────────────
FEATURE_COLS = [
    'rmssd', 'sdnn', 'mean_rr', 'pnn50',
    'hr_mean', 'hr_std', 'rr_skewness', 'rr_kurtosis'
]

X = df[FEATURE_COLS]
y = df['binary_label']
labels_original = df['label']

print(f"\nFeature matrix X: {X.shape}")
print(f"Missing values: {X.isnull().sum().sum()}")

Feature matrix loaded: 8,187 records × 11 columns

Class distribution:
  Normal   (0): 5,042 (61.6%)
  Abnormal (1): 3,145 (38.4%)

Original label breakdown:
label
N    5042
O    2391
A     754

Feature matrix X: (8187, 8)
Missing values: 0


## Train / Validation / Test Split

Three-way stratified split: 80% train, 10% validation, 10% test.

- **Train** — model fitting
- **Validation** — threshold optimisation and model comparison
- **Test** — final held-out evaluation (touched once, after model selection)

Stratification preserves the ~62/38 class ratio across all splits. Random state is fixed for reproducibility.

The original Physionet labels (`N`, `A`, `O`) are carried through for AF-specific secondary analysis — they are NOT used during training.

In [4]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

# ── First split: 80% train, 20% temp ─────────────────────────
X_train, X_temp, y_train, y_temp, labels_train, labels_temp = train_test_split(
    X, y, labels_original,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

# ── Second split: 50/50 of temp → 10% val, 10% test ──────────
X_val, X_test, y_val, y_test, labels_val, labels_test = train_test_split(
    X_temp, y_temp, labels_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=RANDOM_STATE
)

print("Split Summary")
print("="*55)
print(f"{'Set':<12} {'Total':>7} {'Normal':>8} {'Abnormal':>10} {'Ratio':>8}")
print("-"*55)
for name, yy in [('Train', y_train), ('Validation', y_val), ('Test', y_test)]:
    n0 = (yy == 0).sum()
    n1 = (yy == 1).sum()
    print(f"{name:<12} {len(yy):>7,} {n0:>8,} {n1:>10,} {n1/len(yy)*100:>7.1f}%")
print("-"*55)
print(f"{'Total':<12} {len(y):>7,} {(y==0).sum():>8,} {(y==1).sum():>10,} {(y==1).mean()*100:>7.1f}%")

Split Summary
Set            Total   Normal   Abnormal    Ratio
-------------------------------------------------------
Train          6,549    4,033      2,516    38.4%
Validation       819      504        315    38.5%
Test             819      505        314    38.3%
-------------------------------------------------------
Total          8,187    5,042      3,145    38.4%


## Feature Scaling

StandardScaler fitted on training data only. The same scaler is applied to validation and test sets to prevent data leakage.

Scaling is required for Logistic Regression and SVM (distance-based). Random Forest and XGBoost are tree-based and scale-invariant, but we apply scaling uniformly for consistency across all models.

In [5]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=FEATURE_COLS,
    index=X_train.index
)
X_val_scaled = pd.DataFrame(
    scaler.transform(X_val),
    columns=FEATURE_COLS,
    index=X_val.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=FEATURE_COLS,
    index=X_test.index
)

print("Scaling complete — fitted on training set only.")
print(f"\nTrain mean (should be ~0): {X_train_scaled.mean().mean():.6f}")
print(f"Train std  (should be ~1): {X_train_scaled.std().mean():.6f}")

Scaling complete — fitted on training set only.

Train mean (should be ~0): 0.000000
Train std  (should be ~1): 1.000076


## Model Training

Four classifiers trained on scaled training data with `class_weight='balanced'` where supported. This up-weights the minority class (Abnormal, ~38%) during training to improve sensitivity without manual oversampling.

All models use `probability=True` or equivalent to produce probability scores required for threshold optimisation.

| Model | Why included |
|-------|-------------|
| Logistic Regression | Linear baseline — if this passes, the problem is linearly separable |
| Random Forest | Ensemble of decision trees — handles non-linear boundaries |
| XGBoost | Gradient boosting — typically strongest on tabular data |
| SVM (RBF kernel) | Kernel-based — captures complex decision boundaries |

**Calls:** scikit-learn, xgboost

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

try:
    from xgboost import XGBClassifier
    xgb_available = True
except ImportError:
    xgb_available = False
    print("WARNING: xgboost not installed. Run: pip install xgboost")

print("="*55)
print("MODEL TRAINING")
print("="*55)

models = {}

# ── Logistic Regression ───────────────────────────────────────
print("\n  Training Logistic Regression...", end=" ")
lr = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=RANDOM_STATE
)
lr.fit(X_train_scaled, y_train)
models['Logistic Regression'] = lr
print("done.")

# ── Random Forest ─────────────────────────────────────────────
print("  Training Random Forest...", end=" ")
rf = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf.fit(X_train_scaled, y_train)
models['Random Forest'] = rf
print("done.")

# ── XGBoost ───────────────────────────────────────────────────
if xgb_available:
    print("  Training XGBoost...", end=" ")
    # Compute scale_pos_weight for class imbalance
    n_neg = (y_train == 0).sum()
    n_pos = (y_train == 1).sum()
    xgb = XGBClassifier(
        n_estimators=200,
        scale_pos_weight=n_neg / n_pos,
        eval_metric='logloss',
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    xgb.fit(X_train_scaled, y_train)
    models['XGBoost'] = xgb
    print("done.")

# ── SVM (RBF kernel) ─────────────────────────────────────────
print("  Training SVM (RBF)...", end=" ")
svm = SVC(
    kernel='rbf',
    class_weight='balanced',
    probability=True,
    random_state=RANDOM_STATE
)
svm.fit(X_train_scaled, y_train)
models['SVM'] = svm
print("done.")

print(f"\n{len(models)} models trained successfully.")

MODEL TRAINING

  Training Logistic Regression... done.
  Training Random Forest... done.
  Training XGBoost... done.
  Training SVM (RBF)... done.

4 models trained successfully.


## Model Evaluation — Validation Set

Evaluates all trained models on the **validation set** using `src/evaluate.py`. For each model:

1. Finds the optimal classification threshold (maximise specificity while meeting sensitivity ≥ 0.80)
2. Computes sensitivity, specificity, AUROC, F1 (Abnormal class)
3. Runs AF-specific secondary analysis (AF is only 23.9% of Abnormal class — the model may catch Other rhythms better than AF)
4. Documents failure mode if the model does not meet the sensitivity floor

**Calls:** `src/evaluate.evaluate_model()`

In [7]:
from src.evaluate import evaluate_model

print("="*55)
print("MODEL EVALUATION — VALIDATION SET")
print("="*55)

reports = []
for name, model in models.items():
    report = evaluate_model(
        name=name,
        model=model,
        X_test=X_val_scaled,
        y_test=y_val,
        label_col=labels_val
    )
    reports.append(report)

MODEL EVALUATION — VALIDATION SET

  LOGISTIC REGRESSION [PASS]
  Optimal threshold : 0.41
  Sensitivity       : 80.3%
  Specificity       : 84.5%
  AUROC             : 0.8810
  F1 (Abnormal)     : 0.7833
  AF sensitivity    : 95.3%
  Confusion matrix  :
    TP=253  FP=78
    FN=62  TN=426

  RANDOM FOREST [PASS]
  Optimal threshold : 0.37
  Sensitivity       : 80.0%
  Specificity       : 86.7%
  AUROC             : 0.9025
  F1 (Abnormal)     : 0.7950
  AF sensitivity    : 98.8%
  Confusion matrix  :
    TP=252  FP=67
    FN=63  TN=437

  XGBOOST [PASS]
  Optimal threshold : 0.34
  Sensitivity       : 80.0%
  Specificity       : 84.7%
  AUROC             : 0.8874
  F1 (Abnormal)     : 0.7826
  AF sensitivity    : 95.3%
  Confusion matrix  :
    TP=252  FP=77
    FN=63  TN=427

  SVM [PASS]
  Optimal threshold : 0.34
  Sensitivity       : 80.0%
  Specificity       : 88.1%
  AUROC             : 0.8981
  F1 (Abnormal)     : 0.8038
  AF sensitivity    : 98.8%
  Confusion matrix  :
    TP=2

## Model Selection — Natural Selection Framework

Applies the five-criterion selection framework to choose the best model for Layer 2:

1. **Survival condition** — eliminate any model below the sensitivity floor (0.80)
2. **Highest specificity** — among survivors, select the model with highest specificity
3. **AUROC tiebreaker** — if specificity is tied, use AUROC to break the tie
4. **Null outcome protocol** — if no model survives, document as a qualified negative finding

**Calls:** `src/evaluate.select_best_model()`

In [8]:
from src.evaluate import select_best_model

selection = select_best_model(reports, models)

if selection['selected_name']:
    print(f"\nProceeding to held-out test evaluation with: {selection['selected_name']}")
else:
    print(f"\nNo model met the pre-registered threshold.")
    print(f"This is a valid negative finding — document and proceed to conclusions.")


  MODEL SELECTION — NATURAL SELECTION FRAMEWORK

  Surviving models:
    SVM: sensitivity=80.0%  specificity=88.1%  AUROC=0.8981
    Random Forest: sensitivity=80.0%  specificity=86.7%  AUROC=0.9025
    XGBoost: sensitivity=80.0%  specificity=84.7%  AUROC=0.8874
    Logistic Regression: sensitivity=80.3%  specificity=84.5%  AUROC=0.8810

  SELECTED: SVM
  Reason: SVM selected. Met sensitivity floor (80.0%) with highest specificity among survivors (88.1%).

Proceeding to held-out test evaluation with: SVM


## Held-Out Test Evaluation

Final evaluation of the selected model on the **test set** — data the model has never seen during training or threshold optimisation.

This cell is only executed if a model passed the validation selection. The test set is touched exactly once. The threshold from validation is reused — it is NOT re-optimised on the test set.

**Calls:** `src/evaluate.evaluate_model()`

In [9]:
if selection['selected_name']:
    print("="*55)
    print("HELD-OUT TEST EVALUATION")
    print("="*55)

    test_report = evaluate_model(
        name=selection['selected_name'],
        model=selection['selected_model'],
        X_test=X_test_scaled,
        y_test=y_test,
        label_col=labels_test
    )

    print(f"\nTest set result: {'PASS' if test_report['meets_criterion'] else 'FAIL'}")
    if not test_report['meets_criterion']:
        print("Model passed validation but failed on held-out test.")
        print("This may indicate overfitting to the validation set.")
else:
    print("Skipped — no model passed validation selection.")

HELD-OUT TEST EVALUATION

  SVM [PASS]
  Optimal threshold : 0.45
  Sensitivity       : 80.2%
  Specificity       : 91.7%
  AUROC             : 0.9080
  F1 (Abnormal)     : 0.8289
  AF sensitivity    : 96.4%
  Confusion matrix  :
    TP=252  FP=42
    FN=62  TN=463

Test set result: PASS


## Comparison Summary Table

Side-by-side comparison of all models on the validation set. Highlights which models met the pre-registered sensitivity floor and how they compare on secondary metrics.

In [10]:
print("="*75)
print("MODEL COMPARISON — VALIDATION SET")
print("="*75)
print(f"{'Model':<22} {'Sens':>7} {'Spec':>7} {'AUROC':>7} "
      f"{'F1':>7} {'AF Sens':>8} {'Result':>8}")
print("-"*75)

for r in reports:
    af = f"{r['af_sensitivity']:.1%}" if r['af_sensitivity'] is not None else "N/A"
    status = "PASS" if r['meets_criterion'] else "FAIL"
    print(f"{r['name']:<22} "
          f"{r['sensitivity']:>6.1%} "
          f"{r['specificity']:>6.1%} "
          f"{r['auroc']:>7.4f} "
          f"{r['f1_abnormal']:>7.4f} "
          f"{af:>8} "
          f"{status:>8}")

print("="*75)
print(f"\nPre-registered thresholds: Sensitivity ≥ 80.0%, Specificity ≥ 75.0%")

if selection['selected_name']:
    print(f"Selected model: {selection['selected_name']}")
    print(f"Reason: {selection['reason']}")
else:
    print(f"No model selected — qualified negative finding.")

MODEL COMPARISON — VALIDATION SET
Model                     Sens    Spec   AUROC      F1  AF Sens   Result
---------------------------------------------------------------------------
Logistic Regression     80.3%  84.5%  0.8810  0.7833    95.3%     PASS
Random Forest           80.0%  86.7%  0.9025  0.7950    98.8%     PASS
XGBoost                 80.0%  84.7%  0.8874  0.7826    95.3%     PASS
SVM                     80.0%  88.1%  0.8981  0.8038    98.8%     PASS

Pre-registered thresholds: Sensitivity ≥ 80.0%, Specificity ≥ 75.0%
Selected model: SVM
Reason: SVM selected. Met sensitivity floor (80.0%) with highest specificity among survivors (88.1%).


## Save Model and Report

Saves the selected model (via joblib) and the full evaluation report (as JSON) to `outputs/models/`. If no model was selected, only the null outcome report is saved.

**Output:**
- `outputs/models/selected_model.joblib` — fitted sklearn estimator
- `outputs/models/scaler.joblib` — fitted StandardScaler (needed for Layer 2)
- `outputs/models/evaluation_report.json` — full metrics and selection reason

In [11]:
import json
import joblib

MODEL_DIR = r'C:\Projects\GA Capstone Project\outputs\models'
os.makedirs(MODEL_DIR, exist_ok=True)

# ── Save evaluation report ────────────────────────────────────
report_data = {
    'validation_reports': reports,
    'selection': {
        'selected_name':  selection['selected_name'],
        'reason':         selection['reason'],
        'eliminated':     selection['eliminated']
    }
}

if selection['selected_name']:
    report_data['test_report'] = test_report

report_path = os.path.join(MODEL_DIR, 'evaluation_report.json')
with open(report_path, 'w') as f:
    json.dump(report_data, f, indent=2)
print(f"Evaluation report saved: {report_path}")

# ── Save model and scaler ─────────────────────────────────────
if selection['selected_name']:
    model_path = os.path.join(MODEL_DIR, 'selected_model.joblib')
    joblib.dump(selection['selected_model'], model_path)
    print(f"Model saved: {model_path}")

    scaler_path = os.path.join(MODEL_DIR, 'scaler.joblib')
    joblib.dump(scaler, scaler_path)
    print(f"Scaler saved: {scaler_path}")
else:
    print("No model saved — null outcome.")

print("\nNotebook complete. Proceed to 04_apple_watch_bridge.ipynb.")

Evaluation report saved: C:\Projects\GA Capstone Project\outputs\models\evaluation_report.json
Model saved: C:\Projects\GA Capstone Project\outputs\models\selected_model.joblib
Scaler saved: C:\Projects\GA Capstone Project\outputs\models\scaler.joblib

Notebook complete. Proceed to 04_apple_watch_bridge.ipynb.


## Supplementary — Random Forest Feature Importance

Extracts feature importance rankings from the trained Random Forest model. This is a **supplementary analytical finding only** — SVM is the selected Layer 2 model (SVM does not produce native feature importance scores).

RF feature importance shows which of the eight HRV features contributed most to splitting decisions during training. This provides interpretability context for the clinical narrative even though SVM is the deployed model.

**Calls:** nothing from `src/` — inspection and save only
**Output:** `outputs/models/rf_feature_importance.csv`

In [12]:
import pandas as pd
import os

OUTPUTS_DIR = r'C:\Projects\GA Capstone Project\outputs\models'
FEATURE_COLS = [
    'rmssd', 'sdnn', 'mean_rr', 'pnn50',
    'hr_mean', 'hr_std', 'rr_skewness', 'rr_kurtosis'
]

# Extract from trained Random Forest — supplementary analysis
# SVM is the selected Layer 2 model.
# RF feature importance is a secondary analytical finding only.
importance_df = pd.DataFrame({
    'feature':    FEATURE_COLS,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False).reset_index(drop=True)

importance_df['importance_pct'] = (
    importance_df['importance'] * 100
).round(2)

print("Random Forest Feature Importance Rankings")
print("(Supplementary analysis — SVM is Layer 2 model)")
print("="*48)
print(importance_df[['feature','importance_pct']].to_string(
    index=False
))

importance_path = os.path.join(
    OUTPUTS_DIR, 'rf_feature_importance.csv'
)
importance_df.to_csv(importance_path, index=False)
print(f"\nSaved to outputs/models/rf_feature_importance.csv")

Random Forest Feature Importance Rankings
(Supplementary analysis — SVM is Layer 2 model)
    feature  importance_pct
      rmssd           19.58
     hr_std           15.58
    mean_rr           13.32
    hr_mean           12.39
       sdnn           11.87
rr_skewness            9.61
      pnn50            9.31
rr_kurtosis            8.32

Saved to outputs/models/rf_feature_importance.csv


In [13]:
import joblib
import os

OUTPUTS_DIR = r'C:\Projects\GA Capstone Project\outputs\models'

scaler = joblib.load(os.path.join(OUTPUTS_DIR, 'scaler.joblib'))

print("Scaler Verification")
print("="*45)
print(f"Scaler type: {type(scaler).__name__}")
print(f"\nFeature means (fitted on training set only):")
for feat, mean in zip(FEATURE_COLS, scaler.mean_):
    print(f"  {feat:<15}: {mean:.4f}")
print(f"\nIf means look physiologically plausible,")
print(f"scaler was fitted correctly on training data.")

Scaler Verification
Scaler type: StandardScaler

Feature means (fitted on training set only):
  rmssd          : 106.4544
  sdnn           : 88.0448
  mean_rr        : 832.3995
  pnn50          : 0.2563
  hr_mean        : 75.4754
  hr_std         : 10.7064
  rr_skewness    : -0.5065
  rr_kurtosis    : 3.3138

If means look physiologically plausible,
scaler was fitted correctly on training data.
